In [1]:
import fitz
import faiss
import pickle
import numpy as np

from sentence_transformers import SentenceTransformer
from langchain_community.llms import Ollama

C:\Users\Aishwarya\AppData\Local\Temp\ipykernel_38476\120434884.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import Ollama


In [2]:
index = faiss.read_index("../vector_db/bank_index.faiss")

with open("../vector_db/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)

with open("../vector_db/chunk_metadata.pkl", "rb") as f:
    chunk_metadata = pickle.load(f)

print("Knowledge Base Loaded!")
print(index.ntotal)

Knowledge Base Loaded!
66


In [3]:
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

llm = Ollama(model="llama3.1")

print("Models Loaded Successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Models Loaded Successfully!


C:\Users\Aishwarya\AppData\Local\Temp\ipykernel_38476\4109515361.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  llm = Ollama(model="llama3.1")


In [4]:
rbi_pdf = "../data/Regulations/RBI_KYC_Guidelines.pdf"

In [5]:
def extract_text_from_pdf(pdf_path):

    document = fitz.open(pdf_path)

    text = ""

    for page in document:
        text += page.get_text()

    return text

In [6]:
rbi_text = extract_text_from_pdf(rbi_pdf)

print(rbi_text[:1000])

 
 
RBI/DBR/2015-16/18 
Master Direction DBR.AML.BC.No.81/14.01.001/2015-16        
 February 25, 2016 
(Updated as on August 14, 2025) 
(Updated as on June 12, 2025) 
(Updated as on November 06, 2024) 
(Updated as on January 04, 2024) 
(Updated as on October 17, 2023) 
(Updated as on May 04, 2023) 
(Updated as on April 28, 2023) 
 (Updated as on May 10, 2021) 
(Updated as on April 01, 2021) 
(Updated as on March 23, 2021) 
(Updated as on December 18, 2020) 
(Updated as on April 20, 2020) 
(Updated as on April 01, 2020) 
(Updated as on January 09, 2020) 
(Updated as on August 09, 2019) 
(Updated as on May 29, 2019) 
 
Master Direction - Know Your Customer (KYC) Direction, 2016 
Contents 
INTRODUCTION ..................................................................................................................... 3 
CHAPTER  I ............................................................................................................................. 4 
PRELIMINARY .................

In [7]:
rbi_embedding = embedding_model.encode([rbi_text])

rbi_embedding = np.array(rbi_embedding).astype("float32")

In [8]:
distances, indices = index.search(rbi_embedding, k=10)

In [9]:
print("Potentially Impacted Documents:\n")

for idx in indices[0]:
    print(chunk_metadata[idx])

Potentially Impacted Documents:

KYC_Policy.pdf
KYC_Policy.pdf
AML_Policy.pdf
KYC_Policy.pdf
KYC_Policy.pdf
Regulatory Change Management SOP.pdf
AML_Policy.pdf
KYC_Policy.pdf
Customer Onboarding Form.pdf
AML_Policy.pdf


In [10]:
context = "\n\n".join([chunks[idx] for idx in indices[0]])

In [19]:
prompt = f"""
You are a Senior Banking Compliance Officer.

A new RBI circular has been received.

Below is the RBI Circular:

------------------------
{rbi_text[:4000]}
------------------------

Below are the relevant internal bank documents retrieved from the knowledge base:

------------------------
{context}
------------------------

Generate a professional Regulatory Impact Assessment Report.

The report must contain the following sections:

1. Executive Summary

2. Key Regulatory Changes

3. Affected Internal Policies

4. Affected Forms

5. Affected SOPs

6. Risk Assessment
   (Low / Medium / High)

7. Recommended Actions

8. Implementation Priority

Keep the report professional and suitable for submission to a bank compliance team.

Only use the information provided.
"""

In [20]:
report = llm.invoke(prompt)

In [21]:
print(report)

**Regulatory Impact Assessment Report**

**Executive Summary:**
This Regulatory Impact Assessment Report is prepared in response to the newly received RBI circular (RBI/DBR/2015-16/18) dated February 25, 2016, with updated versions up to August 14, 2025. The report aims to assess the impact of the revised Know Your Customer (KYC) Direction on our internal policies, procedures, and risk management framework.

**Key Regulatory Changes:**

1. **Customer Due Diligence (CDD)**: Enhanced CDD measures for high-risk customers, including Politically Exposed Persons (PEPs), non-resident customers from high-risk jurisdictions, cash-intensive businesses, trusts with complex structures, and customers flagged through adverse media or sanctions screening.
2. **Risk-Based Approach**: Adoption of a risk-based approach to categorize customers into low, medium, and high-risk categories based on their transaction volumes, business activities, and other factors.
3. **Periodic KYC Updates**: Requirement for